# Анализ СТП

`input -> document_parser (+ RAG) -> formatter -> output `

In [25]:
from src import Store

FILE_PATH = './data/documents/2. Региональный уровень/СТП Отходы.docx'

store = Store(FILE_PATH)

In [26]:
from src import Model
from pydantic import Field

class QueryModel(Model):
    """
    Информация о запросе пользователя
    """
    settlement : str = Field(description='Населенный пункт')
    location : str = Field(description='Расположение')
    current_year : int = Field(description='Текущий год')

query = QueryModel(
    settlement='г. Гатчина',
    location='Северо-западный федеральный округ, Ленинградская область, Гатчинский муниципальный округ (бывш. Гатчинский муниципальный район)',
    current_year=2026,
)

In [27]:
class ObjectModel(Model):
    """
    Описание объекта
    """
    name : str = Field(description='Наименование объекта')
    location : str = Field(description='Расположение объекта')
    params : str = Field(description='Характеристики объекта')

class SchemaModel(Model):
    """
    Результаты анализа документа схемы территориального планирования
    """
    objects : list[ObjectModel] = Field(min_length=0, description='Перечень объектов, планируемых к размещению')

In [31]:
from src import Agent

agent = Agent(tools=[store.tool], debug=True, response_format=SchemaModel, system_prompt="""
ПРАВИЛА РАБОТЫ:
- ИСПОЛЬЗУЙ инструмент для ответа на вопрос.
- НЕ ПРИДУМЫВАЙ информацию и не пользуйся общими знаниями.
- Если не удается найти информацию, ПРЕКРАЩАЙ поиск.
---
ЗАПРОС:
г. Гатчина, Гатчинский муниципальный район (округ)
""")

res=agent.run([])

[values] {'messages': []}
[updates] {'SummarizationMiddleware.before_model': None}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 685, 'total_tokens': 842, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss:120b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-924', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cfd5b-b4f8-7413-9cbf-7b5bd5c7d0d9-0', tool_calls=[{'name': 'search', 'args': {'filter': None, 'k': 5, 'query': 'Гатчина Гатчинский муниципальный район округ', 'retriever': None, 'search_type': 'similarity'}, 'id': 'call_0m33rbia', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 685, 'output_tokens': 157, 'total_tokens': 842, 'input_token_details': {}, 'output_token_details': {}})]}}
[values] {'messages': [AIMessage(content=

In [33]:
res.model_dump()

{'objects': [{'name': 'Гатчина',
   'location': 'Ленинградская область, Россия',
   'params': 'город областного значения (городской округ) — Гатчинский городской округ (Гатчинский муниципальный округ).'},
  {'name': 'Гатчинский муниципальный район',
   'location': 'Ленинградская область, Россия',
   'params': 'муниципальный район (округ) — Гатчинский муниципальный район, административный центр — г. Гатчина (город, не входящий в состав района, а образующий отдельный городской округ).'}]}